In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aliiihussain/amazon-sales-dataset/amazon_sales_dataset.csv


In [7]:


import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import joblib


df = pd.read_csv("/kaggle/input/datasets/aliiihussain/amazon-sales-dataset/amazon_sales_dataset.csv")


df = df.drop(columns=["review_count","total_revenue"], errors='ignore')




df['rating_class'] = df['rating'].round().astype(int)  


# DATE FEATURES

df["order_date"] = pd.to_datetime(df["order_date"])
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["day"] = df["order_date"].dt.day
df["dayofweek"] = df["order_date"].dt.dayofweek
df["quarter"] = df["order_date"].dt.quarter
df["weekend"] = df["dayofweek"].apply(lambda x: 1 if x>=5 else 0)


# FEATURE ENGINEERING

df["revenue"] = df["price"] * df["quantity_sold"]
df["discount_amount"] = df["price"] * df["discount_percent"] / 100
df["final_price"] = df["price"] - df["discount_amount"]
df["price_per_item"] = df["price"] / (df["quantity_sold"]+1)
df["discount_quantity"] = df["discount_percent"] * df["quantity_sold"]
df["price_discount_interaction"] = df["price"] * df["discount_percent"]
df["revenue_discount"] = df["revenue"] * df["discount_percent"]


# CATEGORICAL FEATURES

cat_cols = ["product_category","customer_region","payment_method"]

# K-Fold Target Encoding to prevent leakage
target = "rating_class"
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_cols:
    df[col+"_te"] = 0
    for train_idx, val_idx in kf.split(df, df[target]):
        df_train, df_val = df.iloc[train_idx], df.iloc[val_idx]
        means = df_train.groupby(col)[target].mean()
        df.iloc[val_idx, df.columns.get_loc(col+"_te")] = df_val[col].map(means)


# FEATURES & TARGET

features = [
    "price","discount_percent","quantity_sold","revenue","discount_amount",
    "final_price","price_per_item","discount_quantity","price_discount_interaction",
    "revenue_discount","year","month","day","dayofweek","quarter","weekend"
] + cat_cols + [c+"_te" for c in cat_cols]

X = df[features]
y = df[target]  # integer classes 1-5

# Original categorical indices for CatBoost
cat_features = [X.columns.get_loc(c) for c in cat_cols]

# TRAINING WITH STRATIFIED K-FOLD

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=8,
        eval_metric='Accuracy',
        cat_features=cat_features,
        verbose=200,
        early_stopping_rounds=100
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val))

    pred = model.predict(X_val)
    acc = accuracy_score(y_val, pred)
    acc_scores.append(acc)

print("CV Accuracy Scores:", acc_scores)
print("Mean CV Accuracy:", np.mean(acc_scores))
print("\nClassification Report for last fold:")
print(classification_report(y_val, pred))


# TRAIN FINAL MODEL ON FULL DATA

final_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    eval_metric='Accuracy',
    cat_features=cat_features,
    verbose=200
)

final_model.fit(X, y)


# FEATURE IMPORTANCE

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": final_model.get_feature_importance()
}).sort_values(by="importance", ascending=False)

print("\nTop 15 Features:")
print(importance.head(15))


# SAVE MODEL

joblib.dump(final_model, "amazon_rating_catboost.pkl")
print("\nFinal model saved as amazon_rating_catboost.pkl")

/tmp/ipykernel_55/234756337.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3.01429216 3.01429216 2.99268548 ... 3.00330628 3.01429216 3.00330628]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[val_idx, df.columns.get_loc(col+"_te")] = df_val[col].map(means)
/tmp/ipykernel_55/234756337.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3.0076808  2.99860321 3.01280512 ... 2.99860321 2.99860321 2.99860321]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[val_idx, df.columns.get_loc(col+"_te")] = df_val[col].map(means)
/tmp/ipykernel_55/234756337.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3.00360114 3.00360114 3.00360114 ... 

0:	learn: 0.2883500	test: 0.2723000	best: 0.2723000 (0)	total: 236ms	remaining: 7m 51s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2784
bestIteration = 20

Shrink model to first 21 iterations.
0:	learn: 0.2925000	test: 0.2766000	best: 0.2766000 (0)	total: 194ms	remaining: 6m 28s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2768
bestIteration = 1

Shrink model to first 2 iterations.
0:	learn: 0.2885250	test: 0.2724000	best: 0.2724000 (0)	total: 192ms	remaining: 6m 22s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2822
bestIteration = 7

Shrink model to first 8 iterations.
0:	learn: 0.2866000	test: 0.2777000	best: 0.2777000 (0)	total: 202ms	remaining: 6m 43s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.283
bestIteration = 9

Shrink model to first 10 iterations.
0:	learn: 0.2935000	test: 0.2758000	best: 0.2758000 (0)	total: 190ms	remaining: 6m 20s
Stopped by overfitting detector  (100 iteration

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0:	learn: 0.2891400	total: 215ms	remaining: 7m 10s
200:	learn: 0.3357600	total: 38.5s	remaining: 5m 44s
400:	learn: 0.3722400	total: 1m 23s	remaining: 5m 33s
600:	learn: 0.4271400	total: 2m 8s	remaining: 4m 59s
800:	learn: 0.4708200	total: 2m 53s	remaining: 4m 19s
1000:	learn: 0.5073800	total: 3m 38s	remaining: 3m 37s
1200:	learn: 0.5423800	total: 4m 22s	remaining: 2m 54s
1400:	learn: 0.5748800	total: 5m 7s	remaining: 2m 11s
1600:	learn: 0.6029400	total: 5m 51s	remaining: 1m 27s
1800:	learn: 0.6304600	total: 6m 35s	remaining: 43.8s
1999:	learn: 0.6560000	total: 7m 19s	remaining: 0us

Top 15 Features:
                feature  importance
12                  day   10.088835
19  product_category_te    9.564022
21    payment_method_te    9.228483
20   customer_region_te    8.950216
11                month    7.810310
13            dayofweek    7.507525
3               revenue    5.193289
6        price_per_item    5.071166
9      revenue_discount    4.088467
10                 year    4.034